# Kelman Filtering

In [1]:
!pip install websocket-client

  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
Using cached websocket_client-1.9.0-py3-none-any.whl (82 kB)


In [21]:
import json
import queue
import threading
import time
import websocket
import numpy as np

In [31]:
class WebSocketCollector:
    def __init__(self, url, max_messages=50):
        self.url = url
        self.max_messages = max_messages
        self.queue = queue.Queue()
        self.received = 0
        self.ws = None

    def on_message(self, ws, message):
        self.queue.put(message)
        self.received += 1
        if self.received >= self.max_messages:
            ws.close()

    def on_error(self, ws, error):
        print("WebSocket error:", error)

    def on_close(self, ws, close_status_code, close_msg):
        print("WebSocket closed")

    def on_open(self, ws):
        print("WebSocket opened")

    def collect(self):
        self.ws = websocket.WebSocketApp(
            self.url,
            on_open=self.on_open,
            on_message=self.on_message,
            on_error=self.on_error,
            on_close=self.on_close,
        )
        thread = threading.Thread(target=self.ws.run_forever, daemon=True)
        thread.start()

        while self.received < self.max_messages:
            time.sleep(0.1)

        thread.join(timeout=1)
        return self.queue

ws_url = "ws://localhost:8000/ws"
message_queue = WebSocketCollector(ws_url, max_messages=50).collect()
print("Collected messages:", message_queue.qsize())

WebSocket opened
WebSocket closed
Collected messages: 50


In [32]:
print("Sample message:", message_queue.get())

raw_values = [message_queue.get() for _ in range(message_queue.qsize())]
values = [json.loads(value).get("data").get("value") for value in raw_values]
values = [float(value) for value in values if value is not None]
print("Extracted values:", values)

Sample message: {"type": "initial_state", "data": {"sensors": [{"id": "f868cfdf-7597-42e6-b592-06bac9b4b17e", "name": "Temperature S1", "type": "gaussian", "interval": 0.1, "min_value": 0, "max_value": 100, "is_running": false, "current_value": 61.095333678064364, "recording": false, "parameters": {}}], "mqtt_connected": false, "recordings": {}}}
Extracted values: [50.800450895606, 57.389965793407356, 48.691535834574935, 53.84765647144533, 43.837765609950765, 22.08820543947434, 41.0808291297256, 73.55642162690579, 41.77448580220914, 17.491119369221117, 4.287760866636741, 34.81313830922739, 67.50895743586798, 47.510328654575396, 60.769082144978725, 62.90306107866573, 44.236932165012234, 54.9151817786044, 39.005658468791005, 69.88708923352397, 54.91907627140688, 62.986527981959625, 39.50464710500468, 33.34225685107889, 36.017292282275875, 60.266643987531246, 32.34616851588815, 38.79777139178323, 11.871203239139213, 75.74140880684776, 26.381940103485224, 29.97924097152663, 10.356894700981

In [33]:
class KalmanFilter1D:
    """
    1D Kalman Filter with state-space representation.
    
    State vector: x = [position, velocity]^T
    
    Process model:
        x_{k|k-1} = F * x_{k-1|k-1} + w_k, where w ~ N(0, Q)
    
    Measurement model:
        z_k = H * x_{k|k-1} + v_k, where v ~ N(0, R)
    
    Parameters:
    -----------
    dt : float
        Time step between measurements
    process_variance : float
        Process noise covariance (Q)
    measurement_variance : float
        Measurement noise covariance (R)
    initial_position : float
        Initial position estimate
    initial_velocity : float
        Initial velocity estimate
    initial_error_covariance : float
        Initial state error covariance P_0
    """
    
    def __init__(self, dt=1.0, process_variance=1e-5, measurement_variance=1e-2, 
                 initial_position=None, initial_velocity=0.0, initial_error_covariance=1.0):
        self.dt = dt
        self.q = process_variance  # Process noise variance
        self.r = measurement_variance  # Measurement noise variance
        
        # State transition matrix (constant velocity model)
        self.F = np.array([[1.0, dt],
                          [0.0, 1.0]])
        
        # Measurement matrix (we only measure position)
        self.H = np.array([[1.0, 0.0]])
        
        # Process noise covariance matrix
        self.Q = np.array([[self.q, 0.0],
                          [0.0, self.q]])
        
        # Measurement noise covariance
        self.R = np.array([[self.r]])
        
        # Initial state: [position, velocity]
        self.x = np.array([[initial_position if initial_position is not None else 0.0],
                          [initial_velocity]])
        
        # Initial state error covariance
        self.P = np.eye(2) * initial_error_covariance
        
        self.estimates = []
        self.velocities = []
        
    def predict(self):
        """Prediction step: x_{k|k-1} = F * x_{k-1|k-1}"""
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        
    def update(self, z):
        """Update step with measurement z"""
        # Innovation (measurement residual)
        y = z - self.H @ self.x
        
        # Innovation covariance
        S = self.H @ self.P @ self.H.T + self.R
        
        # Kalman gain
        K = self.P @ self.H.T / S
        
        # State update
        self.x = self.x + K @ y
        
        # Covariance update
        self.P = (np.eye(2) - K @ self.H) @ self.P
        
    def filter(self, measurements):
        """Apply Kalman filter to a sequence of measurements"""
        self.estimates = []
        self.velocities = []
        
        for measurement in measurements:
            self.predict()
            self.update(np.array([[measurement]]))
            self.estimates.append(self.x[0, 0])
            self.velocities.append(self.x[1, 0])
            
        return self.estimates, self.velocities
    
    def get_state(self):
        """Return current state [position, velocity]"""
        return self.x.flatten()

# Enhanced Kalman filter using class
if 'values' in locals() and values:
    # Use advanced Kalman filter with velocity estimation
    kf = KalmanFilter1D(dt=1.0, 
                       process_variance=1e-5, 
                       measurement_variance=1e-2,
                       initial_position=values[0],
                       initial_velocity=0.0)
    
    filtered_values, velocities = kf.filter(values)
    
    print("=== Advanced Kalman Filter Results ===")
    print(f"Raw values (first 10): {values[:10]}")
    print(f"Filtered values (first 10): {[f'{v:.4f}' for v in filtered_values[:10]]}")
    print(f"Estimated velocities (first 10): {[f'{v:.4f}' for v in velocities[:10]]}")
    print(f"Mean squared error (Filter vs Raw): {np.mean([(f - r)**2 for f, r in zip(filtered_values, values)]):.6f}")


=== Advanced Kalman Filter Results ===
Raw values (first 10): [50.800450895606, 57.389965793407356, 48.691535834574935, 53.84765647144533, 43.837765609950765, 22.08820543947434, 41.0808291297256, 73.55642162690579, 41.77448580220914, 17.491119369221117]
Filtered values (first 10): ['50.8005', '57.2662', '51.2667', '52.7576', '47.4394', '33.3385', '34.2062', '48.5279', '45.9164', '35.5975']
Estimated velocities (first 10): ['0.0000', '6.2809', '-1.0156', '0.0534', '-1.7341', '-5.1137', '-3.7229', '-0.0558', '-0.5196', '-2.1375']
Mean squared error (Filter vs Raw): 222.521056
